In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [ ]:
# !pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.6 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
model = ChatGroq(model="qwen/qwen3.8-27b", api_key=GROQ_API_KEY)

In [ ]:
parser = StrOutputParser()

### 1. Simple Chain

In [ ]:
prompt1 = PromptTemplate(
    template='Write a 3 line on following topic. /n {topic}',
    input_variables=['topic']
)

In [ ]:
simple_chain = prompt1 | model | parser

result = simple_chain.invoke({'topic': 'Python'})
print(result)

Python is a high-level, interpreted programming language renowned for its clean syntax and readability.
It supports multiple programming paradigms, including procedural, object-oriented, and functional styles.
Its extensive standard library and active community make it a versatile choice for web development, data science, and automation.


In [ ]:
!pip install grandalf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.2 MB/s eta 0:00:00


In [ ]:
import importlib
import langchain_core.runnables.graph_ascii as ga

importlib.reload(ga)
print(ga._HAS_GRANDALF)

True


In [ ]:
simple_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


### 2. Sequential Chain

In [ ]:
prompt2 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

In [ ]:
prompt3 = PromptTemplate(
    template='Write a 5 line summary of following text. \n {text}',
    input_variables=['text']
)

In [ ]:
sequential_chain = prompt2 | model | parser | prompt3 | model | parser

result = sequential_chain.invoke({'topic': 'Gen AI'})
print(result)

Generative AI marks a paradigm shift from pattern recognition to content creation, utilizing core architectures like Transformers and diffusion models to produce original text, images, code, and audio. These systems are rapidly transforming industries such as healthcare, finance, and software development by automating routine tasks, accelerating innovation, and democratizing creative capabilities. While the technology promises substantial economic gains and productivity boosts, it simultaneously disrupts labor markets and requires significant workforce upskilling. Significant challenges remain, including ethical concerns over bias and deepfakes, security risks like prompt injection, and the environmental impact of high energy consumption. Consequently, global regulatory frameworks are emerging to address these risks, while future advancements focus on deeper multimodal integration and improved reliability.


In [ ]:
sequential_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

### 3. Parallel Chain

In [ ]:
# !pip install langchain-huggingface
# !pip install -q huggingface_hub

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from huggingface_hub import InferenceClient

In [ ]:
HUGGINGFACEHUB_ACCESS_TOKEN = userdata.get("HF_TOKEN")

In [ ]:
from huggingface_hub import InferenceClient
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

client = InferenceClient(api_key=HF_TOKEN)

candidate_models = [
    "meta-llama/Llama-3.1-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen3.8-27B",
    "deepseek-ai/DeepSeek-V4.1-Flash",
    "google/gemma-2-2b-it",
    "google/gemma-4-31B-it",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "microsoft/Phi-3-mini-4k-instruct",
    "HuggingFaceH4/zephyr-7b-beta",
]

print(f"{'MODEL':<45} {'STATUS':<12} {'FINISH_REASON'}")
print("-" * 75)

for model_name in candidate_models:
    try:
        response = client.chat_completion(
            model=model_name,
            messages=[{"role": "user", "content": "Hi"}],
            max_tokens=20,
        )
        finish = response.choices[0].finish_reason
        content = response.choices[0].message.content
        status = "✅ WORKS" if content else "⚠️ EMPTY"
        print(f"{model_name:<45} {status:<12} {finish}")
    except Exception as e:
        print(f"{model_name:<45} {'❌ FAILED':<12} {str(e)[:60]}")

MODEL                                         STATUS       FINISH_REASON
---------------------------------------------------------------------------
meta-llama/Llama-3.1-8B-Instruct              ✅ WORKS      stop
meta-llama/Llama-3.2-3B-Instruct              ❌ FAILED     (Request ID: Root=1-6abbb6e7-539e2ab57795f86d07f631d8;82358f
Qwen/Qwen2.5-7B-Instruct                      ❌ FAILED     (Request ID: Root=1-6abbb6e7-77daa3fb5de72f9179ef30d8;1a20d7
Qwen/Qwen3.8-27B                              ⚠️ EMPTY     length
deepseek-ai/DeepSeek-V4.1-Flash               ✅ WORKS      stop
google/gemma-2-2b-it                          ❌ FAILED     (Request ID: Root=1-6abbb6e8-2a28b6592827331c7f9816bb;34c69f
google/gemma-4-31B-it                         ✅ WORKS      stop
mistralai/Mistral-7B-Instruct-v0.3            ❌ FAILED     (Request ID: Root=1-6abbb6f2-4c2ad6965907c1073d227901;0311b0
microsoft/Phi-3-mini-4k-instruct              ❌ FAILED     (Request ID: Root=1-6abbb6f2-2485e240208487833430b333;

In [ ]:
# client = InferenceClient(api_key=HUGGINGFACEHUB_ACCESS_TOKEN)

# response = client.chat_completion(
#   model="meta-llama/Llama-3.1-8B-Instruct",
#   messages=[{"role": "user", "content": "Write a detailed report on Gen AI"}],
#   max_tokens=1024,
# )

# print(response.choices[0].message.content)

**General Report on Generalized Artificial Intelligence (Gen AI)**

**Introduction**

Generalized Artificial Intelligence (Gen AI) is a subfield of artificial intelligence (AI) that focuses on creating a single, unified AI system that can perform any intellectual task that a human can. Gen AI aims to develop an AI that can learn, reason, and apply knowledge across a wide range of tasks, much like a human brain. In this report, we will delve into the concept of Gen AI, its history, current state, challenges, and potential applications.

**History of Gen AI**

The concept of Gen AI dates back to the 1950s, when the first AI programs were developed. However, it wasn't until the 1980s that the idea of a single, unified AI system began to take shape. The term "Gen AI" was first coined in the 1990s, and since then, researchers have been working towards creating a Gen AI system.

**Current State of Gen AI**

Currently, Gen AI is still in its infancy, and significant challenges need to be over

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id = "meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token = HUGGINGFACEHUB_ACCESS_TOKEN,
    task = "text-generation",
    max_new_tokens = 1024, # increase generation length
)
model2 = ChatHuggingFace(llm = llm)

In [ ]:
prompt4 = PromptTemplate(
    template='Prepare 5 simple question and its answer from the following text. \n {text}',
    input_variables=['text']
)

In [ ]:
prompt5 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document. \n notes -> {notes} \n quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

In [ ]:
!pip install langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.3 MB/s eta 0:00:00


In [ ]:
from langchain_classic.schema.runnable import RunnableParallel, RunnablePassthrough

In [ ]:
input_chain = prompt2 | model2 | parser
parallel_chain = RunnableParallel({
    'notes': prompt3 | model | parser,
    'quiz': prompt4 | model2 | parser
})
full_chain = (
    input_chain | (lambda report: {"text": report}) | parallel_chain
)
merge_chain = full_chain | prompt5 | model | parser

result = merge_chain.invoke({'topic': 'Gen AI'})
print(result)

Here is the merged document, combining the provided notes and quiz into a cohesive study guide.

***

# General Artificial Intelligence (Gen AI) / AGI: Overview and Study Guide

## Part 1: Core Concepts

**Definition**
General Artificial Intelligence (Gen AI), often referred to as AGI, is a hypothetical AI system designed to perform any intellectual task, adapt to new situations, and potentially surpass human intelligence.

**Key Characteristics**
Gen AI systems are defined by several core attributes:
*   **Autonomy:** The ability to operate independently and make decisions without human intervention.
*   **Self-awareness**
*   **Advanced Reasoning**
*   **Continuous Learning**
*   **Creativity**
*   **Effective Communication**

**Development Technologies**
The development of Gen AI relies on a combination of advanced technologies, including:
*   Deep learning (using neural networks to analyze and interpret complex data)
*   Natural language processing
*   Computer vision
*   Robotics


In [ ]:
merge_chain.get_graph().print_ascii()

                +-------------+                  
                | PromptInput |                  
                +-------------+                  
                        *                        
                        *                        
                        *                        
               +----------------+                
               | PromptTemplate |                
               +----------------+                
                        *                        
                        *                        
                        *                        
              +-----------------+                
              | ChatHuggingFace |                
              +-----------------+                
                        *                        
                        *                        
                        *                        
              +-----------------+                
              | StrOutputParser |                


### 4. Conditional Chain

In [ ]:
from langchain_classic.schema.runnable import RunnableBranch, RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

In [ ]:
class Sentiment(BaseModel):
  sentiment: Literal['positive', 'negative'] = Field(description='give the sentiment of review')

pydantic_parser = PydanticOutputParser(pydantic_object=Sentiment)

In [ ]:
prompt6 = PromptTemplate(
    template='Write a one line review on the following product. \n {topic}',
    input_variables=['topic']
)
prompt7 = PromptTemplate(
    template = 'Classify the sentiment of review text into Positive or Negative. \n {review} \n {format_instruction}',
    input_variables=['review'],
    partial_variables={'format_instruction': pydantic_parser.get_format_instructions()}
)
prompt8 = PromptTemplate(
    template = 'Write one line appropriate respose to this positive feedback. \n {feedback}',
    input_variables=['feedback']
)
prompt9 = PromptTemplate(
    template = 'Write one line appropriate respose to this negative feedback. \n {feedback}',
    input_variables=['feedback']
)

In [ ]:
# review_chain = prompt6 | model | parser
# classifier_chain = review_chain | prompt7 | model | pydantic_parser
# branch_chain = RunnableBranch(
#     (lambda x: x.sentiment == 'positive', prompt8 | model2 | parser),
#     (lambda x: x.sentiment == 'negative', prompt9 | model2 | parser),
#     RunnableLambda(lambda x: 'neutral sentiment')
# )
# final_chain = classifier_chain | branch_chain
# result = final_chain.invoke({'topic': 'Titan Edge Watch'})
# print(result)

review_chain = prompt6 | model | parser
classifier_chain = (lambda x: {'review': x['review']}) | prompt7 | model | pydantic_parser
response_branch = RunnableBranch(
    (lambda x: x['sentiment'].sentiment == 'positive',
     (lambda x: {'feedback': x['review']}) | prompt8 | model | parser),
    (lambda x: x['sentiment'].sentiment == 'negative',
     (lambda x: {'feedback': x['review']}) | prompt9 | model | parser),
    RunnableLambda(lambda x: 'neutral sentiment')
)

full_chain = (
    RunnablePassthrough.assign(review=review_chain)
    | RunnablePassthrough.assign(sentiment=classifier_chain)
    | RunnablePassthrough.assign(response=response_branch)
)

result = full_chain.invoke({'topic': 'Titan Edge Watch'})

print("TOPIC :", result['topic'])
print("REVIEW :", result['review'])
print("SENTIMENT :", result['sentiment'].sentiment)
print("RESPONSE :", result['response'])

TOPIC : Titan Edge Watch
REVIEW : A sleek, rugged timepiece that perfectly balances modern aesthetics with dependable durability.
SENTIMENT : positive
RESPONSE : Thank you for the kind words; I'm thrilled the design strikes that ideal balance between style and strength.


In [ ]:
full_chain.get_graph().print_ascii()

              +-----------------------+            
              | Parallel<review>Input |            
              +-----------------------+            
                   **            ***               
                 **                 **             
               **                     **           
    +----------------+                  **         
    | PromptTemplate |                   *         
    +----------------+                   *         
             *                           *         
             *                           *         
             *                           *         
       +----------+                      *         
       | ChatGroq |                      *         
       +----------+                      *         
             *                           *         
             *                           *         
             *                           *         
   +-----------------+           +-------------+   
   | StrOutp